In [1]:
import sys
!{sys.executable} -m pip install tensorflow torch transformers accelerate scikit-learn

In [1]:
from sklearn.model_selection import train_test_split

texts = [
"I love this app", "Amazing experience", "Very good service", "Excellent app",
"Super fast delivery", "Highly recommended", "Nice interface", "Loved it",
"Very useful", "Great features", "Fantastic service", "Best app ever",

"I hate this app", "Worst experience ever", "Very bad service", "Terrible app",
"Slow delivery", "Not recommended", "Poor interface", "Disappointed",
"Not useful", "Bad features", "Worst service", "Awful app"
]

labels = [1]*12 + [0]*12

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

print("Dataset Loaded ✅")

Dataset Loaded ✅


In [3]:
#RNN IMPLEMENTATION
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=10)
X_test_pad = pad_sequences(X_test_seq, maxlen=10)

X_train_pad = np.array(X_train_pad)
X_test_pad = np.array(X_test_pad)
y_train = np.array(y_train)
y_test = np.array(y_test)

rnn_model = Sequential([
    Embedding(input_dim=1000, output_dim=32, input_length=10),
    SimpleRNN(32),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

rnn_model.fit(X_train_pad, y_train, epochs=5, batch_size=4)

print("RNN Accuracy:", rnn_model.evaluate(X_test_pad, y_test)[1])

C:\Users\mouni\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 5s 19ms/step - accuracy: 0.3158 - loss: 0.7069
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6316 - loss: 0.6816 
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8947 - loss: 0.6668
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8947 - loss: 0.6529
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8947 - loss: 0.6381
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 658ms/step - accuracy: 0.6000 - loss: 0.7128
RNN Accuracy: 0.6000000238418579


In [5]:
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([
    Embedding(input_dim=1000, output_dim=32, input_length=10),
    LSTM(32),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

lstm_model.fit(X_train_pad, y_train, epochs=5, batch_size=4)

print("LSTM Accuracy:", lstm_model.evaluate(X_test_pad, y_test)[1])

Epoch 1/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 18ms/step - accuracy: 0.3158 - loss: 0.6954
Epoch 2/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5789 - loss: 0.6929
Epoch 3/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5263 - loss: 0.6918
Epoch 4/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5789 - loss: 0.6890 
Epoch 5/5
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5789 - loss: 0.6871
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 488ms/step - accuracy: 0.4000 - loss: 0.7031
LSTM Accuracy: 0.4000000059604645


In [9]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

bert_tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

train_enc = bert_tokenizer(X_train, truncation=True, padding=True, max_length=32)

class Dataset(torch.utils.data.Dataset):
    def __init__(self, enc, labels):
        self.enc = enc
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long) for k, v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)  # IMPORTANT FIX
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = Dataset(train_enc, y_train)

bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

optimizer = torch.optim.Adam(bert_model.parameters(), lr=5e-5)

bert_model.train()

for epoch in range(2):
    total_loss = 0

    for i in range(len(train_dataset)):
        batch = train_dataset[i]

        input_ids = batch["input_ids"].unsqueeze(0)
        attention_mask = batch["attention_mask"].unsqueeze(0)
        token_type_ids = batch["token_type_ids"].unsqueeze(0)
        labels = batch["labels"].unsqueeze(0)

        optimizer.zero_grad()

        outputs = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss:", total_loss)

print("BERT Training Done ✅")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 Loss: 14.021490395069122
Epoch 2 Loss: 10.067625895142555
BERT Training Done ✅


In [11]:
import torch
from transformers import RobertaTokenizer, RobertaForSequenceClassification

roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

train_enc = roberta_tokenizer(X_train, truncation=True, padding=True, max_length=32)

train_dataset = Dataset(train_enc, y_train)

roberta_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

optimizer = torch.optim.Adam(roberta_model.parameters(), lr=5e-5)

roberta_model.train()

for epoch in range(2):
    total_loss = 0

    for i in range(len(train_dataset)):
        batch = train_dataset[i]

        input_ids = batch["input_ids"].unsqueeze(0)
        attention_mask = batch["attention_mask"].unsqueeze(0)
        labels = batch["labels"].unsqueeze(0)

        optimizer.zero_grad()

        outputs = roberta_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss:", total_loss)

print("RoBERTa Training Done ✅")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1 Loss: 13.620769679546356
Epoch 2 Loss: 13.887445390224457
RoBERTa Training Done ✅
